Vendas Comercial

In [0]:
%sql
create or replace table medallion.gold.fat_vendas_comercial as

-- Seleciona os detalhes de cada pedido
with itens as (
    select
        i.id_pedido,
        i.id_item,
        i.id_produto,
        p.categoria_produto,
        coalesce(ct.nome_produto_pt, p.categoria_produto) as categoria_produto_pt
    from medallion.silver.fat_itens_pedidos i
    left join medallion.silver.dim_produtos p
        on i.id_produto = p.id_produto
    left join medallion.silver.dim_categoria_produtos_traducao ct
        on p.categoria_produto = ct.nome_produto_pt
),

-- quantidade de itens por pedido (pra dividir o valor)
qtd_itens_pedido as (
    select
        id_pedido,
        count(*) as qtd_itens
    from medallion.silver.fat_itens_pedidos
    group by id_pedido
),

base as (
    select
        year(fp.data_pedido) as ano_venda,
        month(fp.data_pedido) as mes_venda,
        i.categoria_produto_pt as categoria_produto,

        i.id_pedido,
        i.id_item,

        -- dvisiao do valor por item
        (fp.valor_total_pago_brl / q.qtd_itens) as valor_item_brl,
        (fp.valor_total_pago_usd / q.qtd_itens) as valor_item_usd

    from itens i
    join medallion.silver.fat_pedido_total fp
        on i.id_pedido = fp.id_pedido
    join qtd_itens_pedido q
        on i.id_pedido = q.id_pedido
)

select
    ano_venda,
    mes_venda,
    categoria_produto,

    count(distinct id_pedido) as total_pedidos,
    count(*) as qtd_itens_vendidos,

    cast(round(sum(valor_item_brl), 2) as decimal(14,2)) as receita_total_brl,
    cast(round(sum(valor_item_usd), 2) as decimal(14,2)) as receita_total_usd,

    cast(
        round(
            sum(valor_item_brl) / nullif(count(distinct id_pedido), 0),
        2) as decimal(14,2)
    ) as ticket_medio_brl

from base
group by
    ano_venda,
    mes_venda,
    categoria_produto;

num_affected_rows,num_inserted_rows


Rankings Comerciais

    Top 5 Mais Vendidos

In [0]:
from pyspark.sql import functions as F

# Tabelas Silver
df_itens = spark.table("medallion.silver.fat_itens_pedidos")
df_prod = spark.table("medallion.silver.dim_produtos")
df_cat = spark.table("medallion.silver.dim_categoria_produtos_traducao")

# Top 5 produtos MAIS vendidos
top_5_mais = (
    df_itens
    .join(df_prod, "id_produto", "left")
    .join(df_cat, df_prod.categoria_produto == df_cat.nome_produto_pt, "left")
    .withColumn(
        "categoria_produto",
        F.coalesce(F.col("nome_produto_pt"), F.col("categoria_produto"))
    )
    .groupBy("id_produto", "categoria_produto")
    .agg(F.count("*").alias("qtd_vendida"))
    .orderBy(F.desc("qtd_vendida"))
    .limit(5)
)

display(top_5_mais)

id_produto,categoria_produto,qtd_vendida
aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,527
99a4788cb24856965c36a24e339b6058,cama_mesa_banho,488
422879e10f46682990de24d770e7f83d,ferramentas_jardim,484
389d119b48cf3043d311335e499d9c6b,ferramentas_jardim,392
368c6c730842d78016ad823897a372db,ferramentas_jardim,388


    Top 5 Menos Vendidos

In [0]:
# Top 5 produtos MENOS vendidos
top_5_menos = (
    df_itens
    .join(df_prod, "id_produto", "left")
    .join(df_cat, df_prod.categoria_produto == df_cat.nome_produto_pt, "left")
    .groupBy(
        "id_produto",
        F.col("nome_produto_pt").alias("categoria_produto")
    )
    .agg(F.count("*").alias("qtd_vendida"))
    .orderBy(F.asc("qtd_vendida"))
    .limit(5)
)

display(top_5_menos)

id_produto,categoria_produto,qtd_vendida
e9f2586707fb45ea0f9997c54f585842,esporte_lazer,1
e1c0a39b7f806727ea5121c60035ed3c,informatica_acessorios,1
fa11ecd35f999783e96ac500532d9d45,cool_stuff,1
8d7ab3701456fdbfe2526636ce15327a,malas_acessorios,1
cdb8d3c880b6639a70764c6734e6bb69,beleza_saude,1


Avaliação Clientes

In [0]:
%sql
-- Criação da tabela Gold: Avaliações de clientes agregadas por categoria, vendedor e estado
create schema if not exists medallion.gold;
create or replace table medallion.gold.fat_avaliacoes_clientes as

with base as (
    -- Base de pedidos com informações do produto e vendedor
    select distinct
        i.id_pedido,
        v.nome_vendedor,
        p.categoria_produto,
        ct.nome_produto_pt as categoria_produto_pt,
        v.estado as estado_vendedor
    from medallion.silver.fat_itens_pedidos i
    left join medallion.silver.dim_produtos p
        on i.id_produto = p.id_produto
    left join medallion.silver.dim_categoria_produtos_traducao ct
        on p.categoria_produto = ct.nome_produto_pt
    left join medallion.silver.dim_vendedores v
        on i.id_vendedor = v.id_vendedor
),

avaliacoes as (
    -- Avaliações válidas (notas não nulas)
    select
        id_pedido,
        cast(nota_avaliacao as int) as nota
    from medallion.silver.fat_avaliacoes_pedidos
    where nota_avaliacao is not null
)

select
    b.categoria_produto_pt as categoria_produto,
    b.nome_vendedor,
    b.estado_vendedor,
    count(*) as total_avaliacoes,
    round(avg(a.nota), 2) as avaliacao_media,
    sum(case when a.nota >= 4 then 1 else 0 end) as total_avaliacoes_positivas, 
    sum(case when a.nota <= 2 then 1 else 0 end) as total_avaliacoes_negativas, 
    cast(
        round(
            (sum(case when a.nota >= 4 then 1 else 0 end) * 100.0) / nullif(count(*), 0),
            2
        ) as decimal(5,2)
    ) as percentual_satisfacao 
from base b
join avaliacoes a
    on b.id_pedido = a.id_pedido
group by
    b.categoria_produto_pt,
    b.nome_vendedor,
    b.estado_vendedor;

num_affected_rows,num_inserted_rows


 Rankings de Qualidade

In [0]:
# Leitura das tabelas Silver necessárias
df_itens = spark.table("medallion.silver.fat_itens_pedidos")
df_prod = spark.table("medallion.silver.dim_produtos")
df_cat = spark.table("medallion.silver.dim_categoria_produtos_traducao")
df_reviews = spark.table("medallion.silver.fat_avaliacoes_pedidos")
df_vend = spark.table("medallion.silver.dim_vendedores")

# Base combinada: itens + avaliações + produtos + categorias + vendedores
df_base = (
    df_itens
    .join(df_reviews, "id_pedido")  
    .join(df_prod, "id_produto", "left")  
    .join(df_cat, df_prod.categoria_produto == df_cat.nome_produto_pt, "left")  
    .join(df_vend, "id_vendedor", "left")  
    .select(
        "id_pedido",
        "id_produto",
        "nome_produto",
        F.col("nome_produto_pt").alias("categoria_produto"),
        "id_vendedor",
        "nome_vendedor",
        "estado",
        "nota_avaliacao"
    )
    .dropna(subset=["nota_avaliacao"])  # remover notas nulas
)

# ranking de produtos
df_prod_agg = (
    df_base
    .groupBy("id_produto", "nome_produto", "categoria_produto")
    .agg(
        F.count("*").alias("total_avaliacoes"), 
        F.round(F.avg("nota_avaliacao"), 2).alias("nota_media")  
    )
)

# ranking de vendedores
df_vend_agg = (
    df_base
    .groupBy("id_vendedor", "nome_vendedor", "estado")
    .agg(
        F.count("*").alias("total_avaliacoes"),
        F.round(F.avg("nota_avaliacao"), 2).alias("nota_media")
    )
)

    Produto Mais Bem Avaliado

In [0]:
# Produto MAIS bem avaliado (desempate pelo total de avaliações)
produto_melhor = df_prod_agg.orderBy(
    F.desc("nota_media"),
    F.desc("total_avaliacoes")
).limit(1)
display(produto_melhor)

id_produto,nome_produto,categoria_produto,total_avaliacoes,nota_media
37eb69aca8718e843d897aa7b82f462d,Kit de Ferramentas Dourado,ferramentas_jardim,15,5.0


    Produto Menos Bem Avaliado

In [0]:
# Produto MENOS bem avaliado (desempate pelo total de avaliações)
produto_pior = df_prod_agg.orderBy(
    F.asc("nota_media"),
    F.desc("total_avaliacoes")
).limit(1)
display(produto_pior)

id_produto,nome_produto,categoria_produto,total_avaliacoes,nota_media
270516a3f41dc035aa87d220228f844c,Protetor Solar,beleza_saude,10,1.0


    Vendedor Mais Bem Avaliado

In [0]:
# Vendedor MAIS bem avaliado
vendedor_melhor = df_vend_agg.orderBy(
    F.desc("nota_media"),
    F.desc("total_avaliacoes")
).limit(1)
display(vendedor_melhor)

id_vendedor,nome_vendedor,estado,total_avaliacoes,nota_media
48efc9d94a9834137efd9ea76b065a38,Luiz Otávio Abreu,PR,32,5.0


Vendedor Menos Bem Avaliado

In [0]:
# Vendedor MENOS bem avaliado
vendedor_pior = df_vend_agg.orderBy(
    F.asc("nota_media"),
    F.desc("total_avaliacoes")
).limit(1)
display(vendedor_pior)

id_vendedor,nome_vendedor,estado,total_avaliacoes,nota_media
8d92f3ea807b89465643c219455e7369,Sra. Fernanda Santos,SP,8,1.0
